# OpenAI Agents SDK - Brief Demo (FOR DEMO)

Quick demonstration of four key concepts in the OpenAI Agents SDK:
1. **Agents** - Defining and running specialized AI agents
2. **Handoffs** - Agents transferring control to others
3. **Guardrails** - Input/output validation
4. **Sessions** - Stateful conversation history

In [2]:
import os
from dotenv import load_dotenv
from agents import Agent, Runner, SQLiteSession
import asyncio

# Load environment variables
load_dotenv(override=True)

print("✅ OpenAI Agents SDK loaded")

✅ OpenAI Agents SDK loaded


## 1. Agents - Define and Run Specialized AI Agents

In [4]:
# Create specialized agents with specific instructions
MODEL = "gpt-4.1-mini"
math_agent = Agent(
    name="Math Agent",
    instructions="You solve math problems step by step. Be concise.",
    model=MODEL,
)

history_agent = Agent(
    name="History Agent",
    instructions="You answer history questions with dates and context. Be concise.",
    model=MODEL,
)

# Actually run the agents
async def demo_agents():
    # Test math agent
    math_result = await Runner.run(math_agent, "What is 5 + 3?")
    print(f"Math Agent: {math_result.final_output}\n")

    # Test history agent
    history_result = await Runner.run(history_agent, "When was World War II?")
    print(f"History Agent: {history_result.final_output}")

await demo_agents()

Math Agent: 5 + 3 = 8

History Agent: World War II lasted from 1939 to 1945. It began on September 1, 1939, with Germany's invasion of Poland, and ended on September 2, 1945, with Japan's formal surrender.


## 2. Handoffs - Agents Transfer Control

In [6]:
# Triage agent can hand off to specialized agents
triage_agent = Agent(
    name="Triage Agent",
    instructions="Route math questions to Math Agent, history to History Agent. Briefly say who you're routing to.",
    handoffs=[math_agent, history_agent],
    model=MODEL,
)

# Demonstrate actual handoff
async def demo_handoff():
    result = await Runner.run(triage_agent, "What is 10 divided by 2?")
    print(result.last_agent)
    print(f"Triage: {result.final_output}")
await demo_handoff()

# Demonstrate actual handoff
async def demo_handoff2():
    result = await Runner.run(triage_agent, "When did Singapore gain independence?")
    print(result.last_agent)
    print(f"Triage: {result.final_output}")
await demo_handoff2()

Agent(name='Math Agent', handoff_description=None, tools=[], mcp_servers=[], mcp_config={}, instructions='You solve math problems step by step. Be concise.', prompt=None, handoffs=[], model='gpt-4.1-mini', model_settings=ModelSettings(temperature=None, top_p=None, frequency_penalty=None, presence_penalty=None, tool_choice=None, parallel_tool_calls=None, truncation=None, max_tokens=None, reasoning=None, verbosity=None, metadata=None, store=None, include_usage=None, response_include=None, top_logprobs=None, extra_query=None, extra_body=None, extra_headers=None, extra_args=None), input_guardrails=[], output_guardrails=[], output_type=None, hooks=None, tool_use_behavior='run_llm_again', reset_tool_choice=True)
Triage: 10 divided by 2 is calculated as follows:

\[
10 \div 2 = 5
\]

So, the answer is 5.
Agent(name='History Agent', handoff_description=None, tools=[], mcp_servers=[], mcp_config={}, instructions='You answer history questions with dates and context. Be concise.', prompt=None, ha

## 3. Guardrails - Input Validation (Optional)

### 3.1 prompt-based restriction

In [7]:
# Create an agent with validation built into instructions
# This demonstrates guardrail concept through agent behavior
restricted_agent = Agent(
    name="Restricted Agent",
    instructions="""You ONLY answer math or history questions.
    If asked about anything else (weather, sports, food, etc.), respond:
    'I can only help with math or history questions.'"""
)

async def demo_guardrails():
    print("Testing input validation through agent behavior:\n")

    # Valid: Math question
    result = await Runner.run(restricted_agent, "What is 5 + 3?")
    print(f"✅ Math question: {result.final_output}\n")

    # Valid: History question
    result = await Runner.run(restricted_agent, "When was the American Revolution?")
    print(f"✅ History question: {result.final_output}\n")

    # Invalid: Weather question (agent should refuse)
    result = await Runner.run(restricted_agent, "What's the weather today?")
    print(f"❌ Weather question: {result.final_output}")

await demo_guardrails()

Testing input validation through agent behavior:

✅ Math question: 5 + 3 = 8.

✅ History question: The American Revolution took place from 1775 to 1783. This conflict was fought between the thirteen American colonies and Great Britain, resulting in the independence of the United States.

❌ Weather question: I can only help with math or history questions.


### 3.2 Real SDK Input 
This version uses a real SDK **input guardrail**:

```text
User Input
    |
    v
Input Guardrail
    |
    +-- allowed --> Restricted Agent --> Answer
    |
    +-- blocked --> Tripwire --> Execution stops
```

The guardrail uses a small classification agent to determine whether the user's request is about **math or history**. If it is not, `tripwire_triggered=True` causes `InputGuardrailTripwireTriggered`.

`run_in_parallel=False` is used so the guardrail finishes **before** the protected agent starts.

In [13]:
from pydantic import BaseModel

from agents import (
    Agent,
    Runner,
    GuardrailFunctionOutput,
    InputGuardrailTripwireTriggered,
    RunContextWrapper,
    TResponseInputItem,
    input_guardrail,
)

class TopicCheck(BaseModel):
    is_allowed: bool
    reason: str

# A dedicated agent evaluates whether the input is in scope.
guardrail_agent = Agent(
    name="Math or History Guardrail",
    instructions=(
        "Determine whether the user's request is about mathematics or history. "
        "Set is_allowed=true only for math or history questions. "
        "Set is_allowed=false for other topics such as weather, sports, food, travel, etc."
    ),
    output_type=TopicCheck,
    model=MODEL,
)

@input_guardrail
async def math_history_guardrail(
    ctx: RunContextWrapper[None],
    agent: Agent,
    input: str | list[TResponseInputItem],
) -> GuardrailFunctionOutput:

    check = await Runner.run(
        guardrail_agent,
        input,
        context=ctx.context,
    )

    return GuardrailFunctionOutput(
        output_info=check.final_output,
        tripwire_triggered=not check.final_output.is_allowed,
    )

restricted_agent = Agent(
    name="Restricted Agent",
    instructions=(
        "Answer math and history questions clearly and concisely. "
        "Input scope is enforced by an SDK input guardrail."
    ),
    model=MODEL,
    input_guardrails=[math_history_guardrail],  # ✅ CORRECTED
)
async def run_guardrail_test(label: str, question: str):
    print(f"{label}: {question}")

    try:
        result = await Runner.run(restricted_agent, question)
        print(f"✅ Allowed: {result.final_output}\n")

    except InputGuardrailTripwireTriggered as exc:
        info = exc.guardrail_result.output.output_info
        print("🛑 Blocked by input guardrail")
        print(f"Reason: {info.reason}\n")
async def demo_guardrails():
    await run_guardrail_test(
        "Math question",
        "What is 5 + 3?",
    )

    await run_guardrail_test(
        "History question",
        "When was the American Revolution?",
    )

    await run_guardrail_test(
        "Weather question",
        "What's the weather today?",
    )


await demo_guardrails()


Math question: What is 5 + 3?
✅ Allowed: 5 + 3 = 8

History question: When was the American Revolution?
✅ Allowed: The American Revolution took place from 1775 to 1783.

Weather question: What's the weather today?
🛑 Blocked by input guardrail
Reason: The request is about weather, which is not related to mathematics or history.



## 4. Sessions - Stateful Conversation History

In [ ]:
# Sessions maintain conversation history across interactions
# Stateful conversation within the current process/session
async def demo_sessions():
    # Create a session for maintaining state
    session = SQLiteSession("demo_conversation") # it is in memory SQLite

    print("With Sessions - Stateful conversation:\n")

    # First interaction
    result1 = await Runner.run(
        math_agent,
        "Calculate 4 * 4",
        session=session
    )
    print(f"Q1: Calculate 4 * 4")
    print(f"A1: {result1.final_output}\n")

    # Second interaction - agent remembers previous context
    result2 = await Runner.run(
        math_agent,
        "Now multiply that result by 3",
        session=session
    )
    print(f"Q2: Now multiply that result by 3")
    print(f"A2: {result2.final_output}")
    print("\n✅ Session maintains conversation history!")

await demo_sessions()

With Sessions - Stateful conversation:

Q1: Calculate 4 * 4
A1: 4 * 4 = 16

Q2: Now multiply that result by 3
A2: 16 * 3 = 48

✅ Session maintains conversation history!


## Summary

**OpenAI Agents SDK** demonstrated:
- **Agents**: Specialized AI agents processing real requests
- **Handoffs**: Multi-agent workflows with control transfer
- **Guardrails**: Input validation through agent behavior
- **Sessions**: Stateful conversations with automatic history management using SQLiteSession

Each concept was shown with actual running code demonstrating real agent behavior.